# Práctica de la lezione 6: programa tu propio KNN

En esta pratica reconstruyes las dos piezas centrales de un clasificador de los k vecinos más cercanos (KNN, por sus siglas en inglés *K-Nearest Neighbors*): la funzione de distancia y la funzione de voto mayoritario. Trabajas sobre el conjunto de dati de los vinos (`load_wine`), el mismo que usaste en la pratica de la lezione 5, así que puedes comparar directamente cómo se comporta KNN frente a lo que ya conoces de ese conjunto de dati.

Encuentras dos esercizios. Cada uno trae un consegna con suggerimenti, una cella de código con partes marcadas como `# TODO` para que completes, y más abajo una soluzione colapsada por si te atascas. Intenta resolverlo da solo antes de mirarla.

Recuerda: no necesitas ninguna conexión a internet ni ninguna API externa. El conjunto de dati viene incluido en scikit-learn.

## Ejercicio 1: completa la distancia euclídea y el voto mayoritario

Tienes ya preparado el conjunto de dati de los vinos, escalado y dividido en addestramento y prueba. También tienes la funzione `knn_predecir` completa, que llama a otras dos funciones que todavía no existen: `distancia_euclidea` y `votar_lezione_mayoritaria`. Tu tarea es completarlas.

**Suggerimenti:**

- En `distancia_euclidea`, convierte `punto_a` y `punto_b` en arrays de NumPy con `np.array()` y aplica `np.linalg.norm()` sobre su diferencia
- En `votar_lezione_mayoritaria`, usa `Counter` del módulo `collections` sobre la lista de etiquetas recibida y su metodo `most_common(1)` para quedarte con la más frecuente
- `Counter(lista).most_common(1)` devuelve una lista con una sola tupla `(etiqueta, veces_que_aparece)`; necesitas solo la etiqueta, en la posición `[0][0]`
- Si algo falla, ejecuta primero la cella de verificación con un valore de k pequeño y revisa el mensaje de errore: te dice en qué línea está el problema

In [ ]:
import numpy as np
from collections import Counter
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Cargamos el mismo conjunto de datos que en la práctica de la clase 5: los vinos
datos_vino = load_wine()
X_vino, y_vino = datos_vino.data, datos_vino.target

# Escalamos las características, igual que hiciste con el cáncer de mama en clase
escalador_vino = StandardScaler()
X_vino_escalado = escalador_vino.fit_transform(X_vino)

X_train_vino, X_test_vino, y_train_vino, y_test_vino = train_test_split(
    X_vino_escalado, y_vino, test_size=0.2, random_state=42
)

# Preparamos los datos en el formato que espera knn_predecir
datos_entrenamiento_vino = list(zip(X_train_vino, y_train_vino))

print("Puntos de entrenamiento:", len(datos_entrenamiento_vino))
print("Puntos de prueba:", len(X_test_vino))
print("Clases posibles:", set(y_vino))

In [ ]:
def distancia_euclidea(punto_a, punto_b):
    """Distancia en línea recta entre dos puntos."""
    # TODO 1: convierte punto_a y punto_b en arrays de NumPy
    #         y devuelve la norma (np.linalg.norm) de su diferencia
    pass


def votar_clase_mayoritaria(etiquetas_vecinos):
    """Devuelve la etiqueta que más se repite en una lista de etiquetas."""
    # TODO 2: cuenta cuántas veces aparece cada etiqueta con Counter
    #         y devuelve la más frecuente con most_common(1)
    pass


# knn_predecir ya está completa: usa las dos funciones que acabas de programar
def knn_predecir(datos_entrenamiento, punto_nuevo, k):
    """
    datos_entrenamiento: lista de tuplas (caracteristicas, etiqueta)
    punto_nuevo: array con las caracteristicas del punto a clasificar
    k: numero de vecinos que participan en la votacion
    """
    if len(datos_entrenamiento) < k:
        raise ValueError("k no puede superar el número de puntos de entrenamiento")

    distancias = []
    for caracteristicas, etiqueta in datos_entrenamiento:
        distancia = distancia_euclidea(caracteristicas, punto_nuevo)
        distancias.append((distancia, etiqueta))

    distancias.sort(key=lambda par: par[0])
    k_vecinos = distancias[:k]

    votos = [etiqueta for _, etiqueta in k_vecinos]
    return votar_clase_mayoritaria(votos)

Ejecuta la siguiente cella de verificación. Si las dos funciones anteriores están bien completadas, la precisión de tu KNN debería acercarse mucho a la de `KNeighborsClassifier`, igual que viste en lezione con el conjunto de dati del cáncer de mama.

In [ ]:
k = 5

predicciones_propias_vino = [
    knn_predecir(datos_entrenamiento_vino, punto, k) for punto in X_test_vino
]
precision_propia_vino = accuracy_score(y_test_vino, predicciones_propias_vino)

modelo_sklearn_vino = KNeighborsClassifier(n_neighbors=k)
modelo_sklearn_vino.fit(X_train_vino, y_train_vino)
precision_sklearn_vino = accuracy_score(y_test_vino, modelo_sklearn_vino.predict(X_test_vino))

print(f"Precisión de tu KNN:        {precision_propia_vino:.3f}")
print(f"Precisión de scikit-learn:  {precision_sklearn_vino:.3f}")

<details>
<summary>💡 Vedi soluzione</summary>

```python
def distancia_euclidea(punto_a, punto_b):
    """Distancia en línea recta entre dos puntos."""
    # TODO 1: convertimos ambos puntos en arrays de NumPy y calculamos
    # la norma (longitud) del vector diferencia entre ambos
    punto_a = np.array(punto_a)
    punto_b = np.array(punto_b)
    return np.linalg.norm(punto_a - punto_b)


def votar_lezione_mayoritaria(etiquetas_vecinos):
    """Devuelve la etiqueta que más se repite en una lista de etiquetas."""
    # TODO 2: Counter cuenta cuántas veces aparece cada etiqueta,
    # y most_common(1) devuelve la más frecuente junto con su conteo
    return Counter(etiquetas_vecinos).most_common(1)[0][0]
```

Con estas dos funciones, `knn_predecir` queda completa: calcula la distancia de cada punto de addestramento al punto nuevo, se queda con los k más cercanos y vota la lezione mayoritaria entre ellos. El risultato sobre el conjunto de dati de los vinos debería acercarse mucho a la precisión de `KNeighborsClassifier`, tal como comprobaste en lezione con el conjunto de dati del cáncer de mama.

</details>

## Ejercicio 2: mide cómo se ralentiza tu KNN frente al de scikit-learn

En lezione mediste el tiempo de ejecución con el conjunto de dati del cáncer de mama. Ahora repites la medición con los vinos, usando tu propia funzione `knn_predecir` del esercizio anterior. La idea es comprobar, con números concretos, cómo crece el tiempo de cada versión a medida que aumenta el tamaño del conjunto de addestramento.

**Suggerimenti:**

- Usa `time.perf_counter()` antes y después del bloque que quieres medir, y resta ambos valorees para obtener el tiempo transcurrido
- Para tu funzione, recorre `X_test_vino` con un bucle y llama a `knn_predecir(subconjunto, punto, k)` para cada punto
- Para scikit-learn, reentrena el modello con `modello_sklearn_vino.fit(...)` usando solo el subconjunto de tamaño `tamano`, y después mide `modello_sklearn_vino.predict(X_test_vino)`
- El subconjunto de `dati_addestramento_vino` de tamaño `tamano` se obtiene con `dati_addestramento_vino[:tamano]`; el de `X_train_vino` y `y_train_vino`, de la misma forma

In [ ]:
import time

tamanos = [30, 60, 100, len(datos_entrenamiento_vino)]

for tamano in tamanos:
    # TODO 1: crea el subconjunto de datos_entrenamiento_vino de tamaño "tamano"
    subconjunto = ...

    # TODO 2: mide el tiempo que tarda tu knn_predecir en clasificar
    #         todos los puntos de X_test_vino usando "subconjunto"
    inicio = time.perf_counter()
    # ... tu bucle aquí ...
    tiempo_propio = ...

    # TODO 3: reentrena modelo_sklearn_vino con X_train_vino[:tamano] y
    #         y_train_vino[:tamano], y mide el tiempo de su predict()
    #         sobre X_test_vino
    inicio = time.perf_counter()
    # ... reentrenamiento y predicción aquí ...
    tiempo_sklearn = ...

    # TODO 4: imprime, para este tamaño, ambos tiempos
    print(...)

<details>
<summary>💡 Vedi soluzione</summary>

```python
import time

tamanos = [30, 60, 100, len(dati_addestramento_vino)]

for tamano in tamanos:
    # TODO 1: subconjunto de addestramento de este tamaño
    subconjunto = dati_addestramento_vino[:tamano]

    # TODO 2: tiempo de tu funzione knn_predecir
    inicio = time.perf_counter()
    for punto in X_test_vino:
        knn_predecir(subconjunto, punto, k)
    tiempo_propio = time.perf_counter() - inicio

    # TODO 3: tiempo de scikit-learn, reentrenando con el mismo tamaño
    modello_sklearn_vino.fit(X_train_vino[:tamano], y_train_vino[:tamano])
    inicio = time.perf_counter()
    modello_sklearn_vino.predict(X_test_vino)
    tiempo_sklearn = time.perf_counter() - inicio

    # TODO 4: imprimir los risultatos
    print(
        f"Tamaño {tamano}: tu funzione {tiempo_propio:.4f} s, "
        f"scikit-learn {tiempo_sklearn:.4f} s"
    )
```

Si anotas estos risultatos en una tabla, verás que el tiempo de tu funzione crece de forma mucho más pronunciada que el de scikit-learn a medida que aumenta el tamaño del conjunto de addestramento. Esa diferencia es la prueba pratica de todo lo que viste en lezione sobre el coste computacional de KNN: la librería usa estructuras de dati y optimizaciones internas que tu versión, escrita para aprender el algoritmo, no incluye.

</details>

## Resumen: lo que te llevas de esta pratica

- La distancia euclídea entre dos puntos se calcula convirtiéndolos en arrays de NumPy y aplicando `np.linalg.norm()` sobre su diferencia
- El voto mayoritario se resuelve con `Counter` y su metodo `most_common(1)`, que devuelve la etiqueta más frecuente entre los k vecinos
- Tu implementación de KNN funciona igual de bien sobre el conjunto de dati de los vinos que sobre el del cáncer de mama, porque el algoritmo no depende del problema concreto
- El tiempo de ejecución de tu funzione crece mucho más rápido que el de scikit-learn a medida que aumenta el conjunto de addestramento, la prueba pratica del coste computacional de KNN

**Abbreviazioni usate en esta pratica:** KNN (K-Nearest Neighbors, k vecinos más cercanos).